# IR Project 2026 - Full Colab Training Workflow

هذا النوتبوك مخصص لإعادة بناء المشروع من الصفر على Google Colab.

ما الذي سنفعله هنا؟

1. تثبيت المكتبات.
2. جلب المشروع من GitHub أو رفعه كـ ZIP.
3. تحميل MS MARCO Dataset.
4. بناء الفهارس والموديلات داخل `artifacts`.
5. تشغيل Evaluation عادي و Evaluation مع Query Refinement.
6. تحميل وتجربة BERT/Sentence-BERT reranker.
7. حفظ `artifacts` و `reports` على Google Drive.

ملاحظة مهمة: Colab للتدريب وبناء الملفات فقط. وقت المقابلة نشغل المشروع المحلي الذي يقرأ الملفات الجاهزة.

## 0. Runtime

من القائمة اختاري:

`Runtime` → `Change runtime type` → اختاري `GPU` إذا متاح.

GPU ليس ضرورياً لـ BM25 وTF-IDF، لكنه مفيد لتجربة BERT reranking.

In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Current dir:', os.getcwd())

## 1. Get The Project

الطريقة الأفضل إذا المشروع مرفوع على GitHub هي `git clone`.

إذا لم تريدي GitHub، ارفعي ZIP من Files في يسار Colab، ثم استخدمي خيار ZIP الموجود بالأسفل.

In [ ]:
# Option A: clone from GitHub
!rm -rf /content/ir_project
!git clone https://github.com/khaderaldiwani/ir_project.git /content/ir_project
%cd /content/ir_project
!git pull
!pwd
!ls -la

### Optional: ZIP upload instead of GitHub

شغلي هذه الخلية فقط إذا رفعتي `ir_project.zip` يدوياً إلى Colab. إذا استخدمتي GitHub، اتركيها بدون تشغيل.

In [ ]:
# Option B: upload ir_project.zip manually, then uncomment and run:
# !rm -rf /content/ir_project
# !unzip -q /content/ir_project.zip -d /content/
# %cd /content/ir_project
# !pwd
# !ls -la

## 2. Install Dependencies

هذه الخطوة تثبت مكتبات المشروع. `sentence-transformers` مطلوبة فقط لـ BERT reranking.

In [ ]:
!pip -q install -r requirements.txt
!pip -q install sentence-transformers

## 3. Download MS MARCO Files

سيتم تحميل:

- qrels
- queries
- collection archive ثم استخراج `collection.tsv`

حجم الأرشيف حوالي 1GB، لذلك قد يأخذ وقتاً.

In [ ]:
!PYTHONPATH=src python scripts/download_msmarco.py
!ls -lh data/raw/msmarco

## 4. Build Indexes And Artifacts

هذه هي خطوة التدريب/البناء الأساسية. بعدها سيظهر مجلد `artifacts` ويحتوي على:

- `search_index.joblib`
- `documents.sqlite`
- `dataset_metadata.json`

للتجربة السريعة استخدمي `50000`. للنسخة النهائية استخدمي `250000`.

In [ ]:
# Quick run for testing only. Run this if you want to verify the pipeline fast.
# !PYTHONPATH=src python scripts/prepare.py --local-msmarco --max-docs 50000 --max-queries 20 --embedding-dims 64

In [ ]:
# Final run used for the project submission.
!PYTHONPATH=src python scripts/prepare.py --local-msmarco --max-docs 250000 --max-queries 43 --embedding-dims 128
!ls -lh artifacts

## 5. Evaluate Base Retrieval Models

هذه الخطوة تحسب:

- MAP
- nDCG@10
- Precision@10
- Recall

In [ ]:
!PYTHONPATH=src python scripts/evaluate.py --local-msmarco --max-queries 43
!cat artifacts/evaluation_metrics.csv

## 6. Evaluate With Query Refinement

هذه الخطوة تعمل Evaluation بعد تفعيل Query Refinement وتولد ملفاً إضافياً.

In [ ]:
!PYTHONPATH=src python scripts/evaluate.py --local-msmarco --max-queries 43 --refine
!cat artifacts/evaluation_metrics_refined.csv

## 7. Test Normal Search

نجرب البحث قبل BERT للتأكد أن الفهارس تعمل.

In [ ]:
!PYTHONPATH=src python scripts/search.py "what is diabetes treatment" --method bm25 --top-k 5
!PYTHONPATH=src python scripts/search.py "diabetis treatement" --method bm25 --top-k 5 --refine

## 8. Download And Test BERT Reranker

هذه الخطوة لا تعيد تدريب BERT. فقط تحمل نموذج جاهز pretrained:

`sentence-transformers/all-MiniLM-L6-v2`

بعدها BM25 يجلب candidates وBERT يعيد ترتيبها دلالياً.

In [ ]:
!PYTHONPATH=src python scripts/download_bert_model.py
!PYTHONPATH=src python scripts/search.py "what is diabetes treatment" --method bert_rerank --top-k 5

## 9. Save Artifacts And Reports To Google Drive

بعد انتهاء التدريب والتقييم، نحفظ المخرجات على Google Drive.

هذه هي الملفات التي ننقلها للمشروع المحلي قبل العرض.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/drive/MyDrive/ir_project_saved
!mkdir -p /content/drive/MyDrive/ir_project_saved
!cp -r artifacts reports /content/drive/MyDrive/ir_project_saved/
!find /content/drive/MyDrive/ir_project_saved -maxdepth 2 -type f | head -50

## 10. What To Move Back To The Local Project

بعد التحميل من Drive، انقلي هذين المجلدين إلى المشروع المحلي واستبدلي القديم:

- `artifacts`
- `reports`

ثم شغلي محلياً:

```powershell
cd "C:\\Users\\Lenovo\\Desktop\\ir dociment\\ir_project"
.\\run_app.ps1
```

وافتحي:

```text
http://localhost:8501
```

إذا أردت تشغيل BERT محلياً، يجب تثبيت `sentence-transformers` على الجهاز المحلي أيضاً. إذا لم يثبت بسبب الإنترنت، شغلي BERT على Colab فقط، وباقي الطرق تعمل محلياً.

## Interview Explanation

جملة جاهزة للشرح:

> استخدمنا Colab لبناء الفهارس والتقييم. بعد انتهاء التدريب حفظنا الملفات الناتجة داخل artifacts مثل search_index.joblib و documents.sqlite. وقت العرض لا نعيد التدريب، بل يقرأ المشروع هذه الملفات مباشرة. كما أضفنا BERT كـ reranker، حيث يسترجع BM25 المرشحين أولاً ثم يعيد BERT ترتيبهم دلالياً بدون حساب BERT لكل الوثائق.